[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

# UofT FASE ML Bootcamp
#### Friday June 11, 2026
#### Diffusion - Lab 2, Day 4
#### Teaching team: Eldan Cohen, Alex Olson, Hriday Chheda
##### Lab author: Hriday Chheda

## Diffusion Models for Language

Diffusion models are a family of generative AI models that learn by reversing a corruption process.

The central idea is surprisingly simple:

1. Start with clean data.
2. Gradually corrupt it.
3. Train a model to recover the original data from the corrupted version.

Once a model learns how to reverse the corruption process, it can generate new content by starting from a highly corrupted state and progressively reconstructing meaningful information.

For images, the corruption process typically involves adding random Gaussian noise to pixels. Language, however, is discrete rather than continuous, so we cannot simply add a small amount of noise to a word. Instead, we corrupt text by masking tokens.

## Diffusion vs Traditional Language Models

Most modern language models, such as GPT, generate text from left to right:

    The
    ↓
    The capital
    ↓
    The capital of
    ↓
    The capital of France
    ↓
    ...

Diffusion language models work differently.

They begin with a partially corrupted sequence and repeatedly refine it:

    [MASK] capital of [MASK] is Paris ...
    ↓
    The capital of [MASK] is Paris ...
    ↓
    The capital of France is Paris ...
    ↓
    The capital of France is Paris and many tourists visit annually.

Rather than predicting the next token, they predict missing tokens and progressively improve the entire sequence.

---

## Connection to LLaDA

LLaDA (Large Language Diffusion with mAsking) is a recent diffusion-based language model. [Link](https://ml-gsai.github.io/LLaDA-demo/)

Instead of generating text one token at a time, it starts with a heavily masked sequence and iteratively reconstructs the missing information. The process is conceptually similar to image diffusion models, but uses masking as the corruption mechanism instead of Gaussian noise.

In this notebook, we will use a masked language model to visualize this process and observe how a sentence gradually emerges as masks are repeatedly replaced with model predictions.

## 1. Setup

We use Hugging Face Transformers and a masked language model.

Unlike a GPT-style model, BERT is trained to predict missing words inside a sentence.

In [ ]:
# If running in a fresh environment, uncomment this line:
# !pip install transformers torch pandas matplotlib

import torch
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForMaskedLM

## 2. Load a masked language model

We will use `bert-base-uncased`.

Important detail: BERT can technically predict special tokens such as `[UNK]`, `[CLS]`, or `[SEP]`.  
For this demo, we explicitly filter those out so the model only chooses normal word tokens.

In [ ]:
MODEL_NAME = "bert-base-uncased"

# The tokenizer converts text into token IDs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# The model predicts a distribution over vocabulary tokens at each [MASK] position.
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model.eval()

print("Mask token:", tokenizer.mask_token)
print("Mask token ID:", tokenizer.mask_token_id)
print("Number of special token IDs:", len(tokenizer.all_special_ids))

## 3. One sentence, corrupted by masking

This is our entire demo example.

Clean sentence:

> The capital of France is Paris and many tourists visit annually.

Corrupted sentence:

> The capital of `[MASK]` is Paris and many `[MASK]` visit annually.

The reverse diffusion process will fill in one mask at a time.

In [ ]:
original_sentence = "The capital of France is Paris and many tourists visit annually."

corrupted_sentence = "[MASK] capital of [MASK] is Paris and many [MASK] visit annually."

print("Original sentence:")
print(original_sentence)

print("Corrupted sentence:")
print(corrupted_sentence)

## 4. Helper function: predict one mask

This function does the core work:

1. Find the first `[MASK]` in the sentence.
2. Ask BERT for token probabilities at that position.
3. Filter out special tokens like `[UNK]`.
4. Replace the mask with the highest-probability valid token.
5. Store the top predictions for inspection.

In [ ]:
def predict_first_mask(text, top_k=5):
    """
    Predict the first [MASK] token in a sentence.

    Returns
    -------
    chosen_token : str
        The selected token used to replace the mask.
    confidence : float
        The model probability for the selected token.
    top_predictions : list of tuples
        A list like [(token, probability), ...] for the top valid predictions.
    """

    if tokenizer.mask_token not in text:
        return None, None, None

    # Convert the text into model inputs.
    inputs = tokenizer(text, return_tensors="pt")

    # Find all positions where the token is [MASK].
    mask_positions = torch.where(
        inputs["input_ids"][0] == tokenizer.mask_token_id
    )[0]

    # We fill only the first mask in this simple demo.
    mask_pos = mask_positions[0]

    with torch.no_grad():
        outputs = model(**inputs)

    # logits shape: [batch_size, sequence_length, vocabulary_size]
    logits = outputs.logits

    # Extract the logits for the masked position.
    mask_logits = logits[0, mask_pos]

    # Convert logits into probabilities.
    probs = torch.softmax(mask_logits, dim=-1)

    # BERT may assign high probability to special tokens such as [UNK].
    # We remove those from consideration.
    special_token_ids = set(tokenizer.all_special_ids)

    # Sort all vocabulary IDs from most likely to least likely.
    sorted_token_ids = torch.argsort(probs, descending=True)

    valid_predictions = []

    for token_id_tensor in sorted_token_ids:
        token_id = token_id_tensor.item()

        # Skip special tokens like [UNK], [CLS], [SEP], [PAD], [MASK].
        if token_id in special_token_ids:
            continue

        token = tokenizer.decode([token_id]).strip()
        confidence = probs[token_id].item()

        # Skip empty strings just to be safe.
        if token == "":
            continue

        valid_predictions.append((token, confidence))

        if len(valid_predictions) >= top_k:
            break

    chosen_token, confidence = valid_predictions[0]

    return chosen_token, confidence, valid_predictions

## 5. Helper function: one denoising step

This replaces the first `[MASK]` with the model's chosen prediction.

In [ ]:
def denoise_once(text, top_k=5):
    """
    Fill one [MASK] token using BERT.

    This is one reverse-diffusion step in our toy text demo.
    """

    chosen_token, confidence, top_predictions = predict_first_mask(text, top_k=top_k)

    if chosen_token is None:
        return text, {
            "chosen_token": None,
            "confidence": None,
            "top_predictions": None,
        }

    # Replace only the first mask.
    new_text = text.replace(tokenizer.mask_token, chosen_token, 1)

    info = {
        "chosen_token": chosen_token,
        "confidence": confidence,
        "top_predictions": top_predictions,
    }

    return new_text, info

## 6. Run the diffusion-style denoising process

We start with the corrupted sentence and fill one mask at a time.

At each step, we save:

- the current sentence
- how many masks remain
- which token was chosen
- the model's confidence
- the top predictions for the chosen mask

In [ ]:
current_text = corrupted_sentence
history = []

step = 0

while True:
    masks_remaining = current_text.count(tokenizer.mask_token)

    # Stop once the sentence has no masks left.
    if masks_remaining == 0:
        history.append({
            "step": step,
            "masks_remaining": masks_remaining,
            "sentence_before_step": current_text,
            "chosen_token": None,
            "confidence": None,
            "top_predictions_for_chosen_mask": None,
        })
        break

    new_text, info = denoise_once(current_text, top_k=5)

    top_pred_text = ", ".join([
        f"{tok} ({prob:.3f})" for tok, prob in info["top_predictions"]
    ])

    history.append({
        "step": step,
        "masks_remaining": masks_remaining,
        "sentence_before_step": current_text,
        "chosen_token": info["chosen_token"],
        "confidence": info["confidence"],
        "top_predictions_for_chosen_mask": top_pred_text,
    })

    current_text = new_text
    step += 1

history_df = pd.DataFrame(history)
history_df

## 7. Visualize the sentence evolving

This table shows the sentence after each denoising step.

The highlighted words are the tokens inserted by the model.

In [ ]:
def make_visual_history(history_df):
    """
    Create a simple HTML visualization of the iterative denoising process.
    """
    rows = []

    for _, row in history_df.iterrows():
        step = int(row["step"])
        sentence = row["sentence_before_step"]
        chosen = row["chosen_token"]
        confidence = row["confidence"]

        if chosen is None:
            chosen_text = "Done"
            conf_text = ""
        else:
            chosen_text = f"<b>{chosen}</b>"
            conf_text = f"{confidence:.3f}"

        # Highlight masks for readability.
        sentence_html = sentence.replace(
            tokenizer.mask_token,
            '<span style="background-color:#ffe08a; padding:2px 4px; border-radius:4px;">[MASK]</span>'
        )

        rows.append(f"""
        <tr>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{step}</td>
            <td style="padding:8px; border-bottom:1px solid #ddd; font-family:monospace;">{sentence_html}</td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{chosen_text}</td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{conf_text}</td>
        </tr>
        """)

    html = f"""
    <table style="border-collapse:collapse; width:100%; font-size:15px;">
        <tr style="background-color:#f2f2f2;">
            <th style="text-align:left; padding:8px;">Step</th>
            <th style="text-align:left; padding:8px;">Sentence before denoising step</th>
            <th style="text-align:left; padding:8px;">Token inserted</th>
            <th style="text-align:left; padding:8px;">Confidence</th>
        </tr>
        {''.join(rows)}
    </table>
    """

    return HTML(html)

display(make_visual_history(history_df))

## 8. Final result

Compare the original sentence and the reconstructed sentence.

In [ ]:
final_sentence = history_df.iloc[-1]["sentence_before_step"]

print("Original sentence:")
print(original_sentence)

print("Corrupted sentence:")
print(corrupted_sentence)

print("Reconstructed sentence:")
print(final_sentence)

## Key takeaway

```text
Forward process:
clean text → masked/corrupted text

Reverse process:
masked/corrupted text → progressively reconstructed text
```

In this notebook, BERT is acting as our denoising model.

A full diffusion language model, such as LLaDA-style models, applies this idea at scale: generate text by repeatedly denoising masked tokens rather than generating strictly left-to-right.